# План анализа: заполняемость поля "Зона ответственности"

Шаблон тетрадки для сравнения 3 источников:
1. Excel-отчет
2. Таблица на озере (Lake #1)
3. Таблица на озере (Lake #2)

Цель: сравнить полноту заполнения поля `Зона ответственности`, а также расхождения по значениям между источниками.

## 0) Настройки и входные параметры

Заполните пути/запросы и названия колонок под ваши реальные данные.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

# ===== 1) Источник Excel =====
EXCEL_PATH = Path(r"PUT_YOUR_EXCEL_PATH_HERE.xlsx")
EXCEL_SHEET = "Sheet1"  # при необходимости измените

# ===== 2) Источник Lake #1 =====
# Вариант A: если выгрузка уже в csv/parquet
LAKE1_FILE = Path(r"PUT_YOUR_LAKE1_FILE_HERE.csv")
# Вариант B: если будете читать SQL (оставьте как шаблон)
LAKE1_SQL = """-- PUT YOUR SQL HERE"""

# ===== 3) Источник Lake #2 =====
LAKE2_FILE = Path(r"PUT_YOUR_LAKE2_FILE_HERE.csv")
LAKE2_SQL = """-- PUT YOUR SQL HERE"""

# Общий ключ для сопоставления записей между источниками
# Например: ИНН, client_id, договор_id и т.д.
KEY_COL = "inn"

# Названия колонок "Зона ответственности" в каждом источнике
ZONE_COLS = {
    "excel": "zone_resp_excel",
    "lake1": "zone_resp_lake1",
    "lake2": "zone_resp_lake2",
}

print("Шаблон параметров загружен. Заполните пути и имена колонок.")

## 1) Загрузка данных

Ниже блоки-шаблоны для чтения данных. Оставьте только нужные варианты.

In [ ]:
# ===== Excel =====
# Подстройте dtype/usecols при необходимости
df_excel_raw = pd.read_excel(EXCEL_PATH, sheet_name=EXCEL_SHEET)

# ===== Lake #1 (файл) =====
if LAKE1_FILE.suffix.lower() == ".parquet":
    df_lake1_raw = pd.read_parquet(LAKE1_FILE)
else:
    df_lake1_raw = pd.read_csv(LAKE1_FILE)

# ===== Lake #2 (файл) =====
if LAKE2_FILE.suffix.lower() == ".parquet":
    df_lake2_raw = pd.read_parquet(LAKE2_FILE)
else:
    df_lake2_raw = pd.read_csv(LAKE2_FILE)

# Если будете читать из БД/озера по SQL, замените блоки выше на свой коннектор.
# Пример:
# from sqlalchemy import create_engine
# engine = create_engine("...")
# df_lake1_raw = pd.read_sql(LAKE1_SQL, engine)

print("Excel:", df_excel_raw.shape)
print("Lake1:", df_lake1_raw.shape)
print("Lake2:", df_lake2_raw.shape)

## 2) Приведение к единому формату

Цель: в каждом источнике оставить `KEY_COL` и колонку зоны в одинаковом формате.

In [ ]:
def normalize_text(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip()
    if s == "":
        return np.nan
    return s


def prepare_source(df_raw: pd.DataFrame, key_col: str, zone_col: str, source_name: str) -> pd.DataFrame:
    cols_missing = [c for c in [key_col, zone_col] if c not in df_raw.columns]
    if cols_missing:
        raise KeyError(f"{source_name}: отсутствуют колонки {cols_missing}")

    out = df_raw[[key_col, zone_col]].copy()
    out = out.rename(columns={key_col: "key_id", zone_col: "zone_resp"})
    out["key_id"] = out["key_id"].astype(str).str.strip()
    out["zone_resp"] = out["zone_resp"].apply(normalize_text)
    out["source"] = source_name
    return out


df_excel = prepare_source(df_excel_raw, KEY_COL, ZONE_COLS["excel"], "excel")
df_lake1 = prepare_source(df_lake1_raw, KEY_COL, ZONE_COLS["lake1"], "lake1")
df_lake2 = prepare_source(df_lake2_raw, KEY_COL, ZONE_COLS["lake2"], "lake2")

for name, frame in [("excel", df_excel), ("lake1", df_lake1), ("lake2", df_lake2)]:
    print(name, frame.shape, "unique keys:", frame["key_id"].nunique())

## 3) Метрики заполняемости по каждому источнику

In [ ]:
def completeness_metrics(df: pd.DataFrame, source_name: str) -> pd.DataFrame:
    total_rows = len(df)
    total_keys = df["key_id"].nunique()
    filled_rows = df["zone_resp"].notna().sum()
    empty_rows = df["zone_resp"].isna().sum()

    key_filled = df.groupby("key_id")["zone_resp"].apply(lambda s: s.notna().any())
    keys_with_zone = key_filled.sum()
    keys_without_zone = (~key_filled).sum()

    return pd.DataFrame([
        {
            "source": source_name,
            "rows_total": total_rows,
            "rows_filled": int(filled_rows),
            "rows_empty": int(empty_rows),
            "rows_fill_rate": round(filled_rows / total_rows, 4) if total_rows else np.nan,
            "keys_total": int(total_keys),
            "keys_with_zone": int(keys_with_zone),
            "keys_without_zone": int(keys_without_zone),
            "keys_fill_rate": round(keys_with_zone / total_keys, 4) if total_keys else np.nan,
        }
    ])


summary = pd.concat([
    completeness_metrics(df_excel, "excel"),
    completeness_metrics(df_lake1, "lake1"),
    completeness_metrics(df_lake2, "lake2"),
], ignore_index=True)

display(summary)

## 4) Сравнение заполняемости по общему набору ключей

Смотрим пересечение ключей и где поле заполнено/пусто в разных источниках.

In [ ]:
excel_key = df_excel.groupby("key_id", as_index=False)["zone_resp"].first().rename(columns={"zone_resp": "zone_excel"})
lake1_key = df_lake1.groupby("key_id", as_index=False)["zone_resp"].first().rename(columns={"zone_resp": "zone_lake1"})
lake2_key = df_lake2.groupby("key_id", as_index=False)["zone_resp"].first().rename(columns={"zone_resp": "zone_lake2"})

cmp = excel_key.merge(lake1_key, on="key_id", how="outer").merge(lake2_key, on="key_id", how="outer")

for col in ["zone_excel", "zone_lake1", "zone_lake2"]:
    cmp[f"filled_{col}"] = cmp[col].notna()

presence_matrix = (
    cmp
    .groupby(["filled_zone_excel", "filled_zone_lake1", "filled_zone_lake2"], as_index=False)
    .size()
    .rename(columns={"size": "keys_count"})
    .sort_values("keys_count", ascending=False)
)

display(presence_matrix)
print("Всего ключей в объединении:", cmp["key_id"].nunique())

## 5) Расхождения значений "Зона ответственности"

Проверяем только ключи, где зона заполнена минимум в двух источниках.

In [ ]:
def compare_pair(df_cmp: pd.DataFrame, left_col: str, right_col: str, pair_name: str) -> pd.DataFrame:
    sub = df_cmp[["key_id", left_col, right_col]].copy()
    sub = sub[sub[left_col].notna() & sub[right_col].notna()].copy()
    sub["is_equal"] = sub[left_col] == sub[right_col]
    res = pd.DataFrame([
        {
            "pair": pair_name,
            "keys_with_both": len(sub),
            "equal_count": int(sub["is_equal"].sum()),
            "diff_count": int((~sub["is_equal"]).sum()),
            "equal_rate": round(sub["is_equal"].mean(), 4) if len(sub) else np.nan,
        }
    ])
    diff_sample = sub[~sub["is_equal"]].head(100)
    return res, diff_sample


pair1_summary, pair1_diff = compare_pair(cmp, "zone_excel", "zone_lake1", "excel_vs_lake1")
pair2_summary, pair2_diff = compare_pair(cmp, "zone_excel", "zone_lake2", "excel_vs_lake2")
pair3_summary, pair3_diff = compare_pair(cmp, "zone_lake1", "zone_lake2", "lake1_vs_lake2")

pair_summary = pd.concat([pair1_summary, pair2_summary, pair3_summary], ignore_index=True)
display(pair_summary)

print("Примеры расхождений excel_vs_lake1:")
display(pair1_diff.head(20))

## 6) Выгрузка результатов анализа (опционально)

Если нужно сохранить результаты в Excel — используйте этот блок.

In [ ]:
OUT_FILE = Path("zone_responsibility_comparison.xlsx")

with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    summary.to_excel(writer, sheet_name="completeness_summary", index=False)
    presence_matrix.to_excel(writer, sheet_name="presence_matrix", index=False)
    pair_summary.to_excel(writer, sheet_name="pair_summary", index=False)
    pair1_diff.to_excel(writer, sheet_name="diff_excel_lake1", index=False)
    pair2_diff.to_excel(writer, sheet_name="diff_excel_lake2", index=False)
    pair3_diff.to_excel(writer, sheet_name="diff_lake1_lake2", index=False)

print(f"Сохранено: {OUT_FILE.resolve()}")

## 7) Чек-лист интерпретации

- Сравните `rows_fill_rate` и `keys_fill_rate` по источникам.
- Проверьте `presence_matrix`: где поле заполнено только в одном источнике.
- По `pair_summary` оцените согласованность значений между источниками.
- Из `diff_*` выгрузок сформируйте правила нормализации/маппинга значений зоны.